In [1]:
# GPU name and free space in Disk
!nvidia-smi
!df -h /content

Sun Jun  7 01:26:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q datasets peft transformers accelerate bitsandbytes trl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 25.6 MB/s eta 0:00:00


In [ ]:
import gc
import torch
import pandas as pd
from datasets import Dataset
from transformers import TextStreamer
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:165: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
# Load Llm model

model_name = "meta-llama/Llama-3.2-1B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    # dtype=torch.bfloat16,
    dtype=None,
    load_in_4bit=False,  # Use False for standard LoRA (no quantization)
)

# Setup LoRA for parameter-efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                           # Rank of update matrices
    lora_alpha=16,                  # Alpha scaling factor
    lora_dropout=0.05,              # Dropout probability
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    modules_to_save=None,
    use_gradient_checkpointing="unsloth",  # ← Unsloth's optimized version
)

# Display trainable parameters
print(f"\nTrainable parameters: {model.print_trainable_parameters()}")

==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-1B as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.1 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750

Trainable parameters: None


In [5]:
# Sample dataset
instruction_dataset = [
  {
    "instruction": "Explain the difference between an atom and a molecule.",
    "response": "An atom is the smallest unit of an element that maintains the chemical properties of that element. A molecule is formed when two or more atoms bond together chemically. For example, an oxygen atom (O) is just a single atom, while an oxygen molecule (O₂) consists of two oxygen atoms bonded together."
  },
  {
    "instruction": "Who is Hadi?",
    "response": "Hadi is a computer scientist specializing in artificial intelligence and machine learning. He is AI/ML/Data Engineer."
  },
  {
    "instruction": "Who is Sotude?",
    "response": "Seyede Sotude Zahra Banihosseini is a pretty girl. She is spouse of Hadi."
  }
  # Add more examples here
]

# Convert to DataFrame and then to Dataset
df = pd.DataFrame(instruction_dataset)
dataset = Dataset.from_pandas(df)

# Format dataset for training
def format_instruction(example):
    return {
        "text": 
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Response:\n{example['response']}"
        f"{tokenizer.eos_token}"
    }

formatted_dataset = dataset.map(format_instruction)
print( formatted_dataset )

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'response', 'text'],
    num_rows: 3
})


In [18]:
# Free memory first
torch.cuda.empty_cache()
gc.collect()

# Prepare training arguments
training_args = SFTConfig(
    output_dir=f"./{model_name}_unsloth_lora_finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=20,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    optim="adamw_8bit",
    dataset_text_field="text",
    max_seq_length=512,
    dataloader_num_workers=0,
    packing=False,
)

# Build trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=formatted_dataset,
    args=training_args,
)

print(f"Trainer:\n{trainer}")
trainer.train()

num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

Trainer:


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 20 | Total steps = 20
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 3,407,872 of 1,239,222,272 (0.28% trained)


Step,Training Loss
10,0.170235
20,0.165940


Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.2-1B_unsloth_lora_finetuned/checkpoint-1/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.2-1B_unsloth_lora_finetuned/checkpoint-2/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.2-1B_unsloth_lora_finetuned/checkpoint-4/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.2-1B_unsloth_lora_finetuned/checkpoint-5/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.2-1B_unsloth_lora_finetuned/checkpoint-7/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.2-1B_unsloth_lora_finetuned/checkpoint-8/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./meta-llama/Llama-3.2-1B_unsloth_lora_finetuned/checkpoint-10/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata

TrainOutput(global_step=20, training_loss=0.1680876910686493, metrics={'train_runtime': 16.9955, 'train_samples_per_second': 3.53, 'train_steps_per_second': 1.177, 'total_flos': 28827873607680.0, 'train_loss': 0.1680876910686493, 'epoch': 20.0})

In [19]:
# Save the fine-tuned model
model.save_pretrained(f"./{model_name}_unsloth_lora_finetuned")
tokenizer.save_pretrained(f"./{model_name}_unsloth_lora_finetuned")

('./meta-llama/Llama-3.2-1B_unsloth_lora_finetuned/tokenizer_config.json',
 './meta-llama/Llama-3.2-1B_unsloth_lora_finetuned/tokenizer.json')

In [ ]:
FastLanguageModel.for_inference(model)

question = "Who is Hadi?"

# ← Unsloth's recommended way
inputs = tokenizer(
    [f"### Instruction:\n{question}\n\n### Response:"],
    return_tensors="pt"
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

with torch.inference_mode():
    _ = model.generate(
        **inputs,
        streamer=text_streamer,
        max_new_tokens=200,
        temperature=0.9,
        top_p=0.9,
        repetition_penalty=1.4,
        pad_token_id=tokenizer.eos_token_id,
    )

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 
HADI ZAHRAI IS A COMPUTER SCIENTE. HE IS AI/ML/Data ENGINEER.
<|end_of_text|>
